In [1]:
import os
import numpy as np
import pandas as pd
import torch  # PyTorch required for ProtBERT
import tensorflow as tf
import time  # For time tracking
from tqdm import tqdm  # For progress bar
from transformers import AutoTokenizer, AutoModel
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, LSTM, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib  # For saving/loading scaler

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load ProtBERT tokenizer and model (Use PyTorch instead of TensorFlow)
MODEL_NAME = "Rostlab/prot_bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME, from_tf=False).to(device)  # Move model to GPU

# Function to extract ProtBERT embeddings with progress bar and GPU utilization
def extract_protbert_features(sequence):
    sequence = " ".join(sequence)  # Space between amino acids
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)  # Move to GPU
    
    start_time = time.time()
    with torch.no_grad():  # Disable gradients for faster inference
        outputs = bert_model(**inputs)

    end_time = time.time()
    
    embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # Extract CLS token representation and move back to CPU
    return embeddings.flatten(), end_time - start_time  # Return features and time taken

# Function to load sequences from CSV files
def load_sequences_from_csv(file_path, label):
    df = pd.read_csv(file_path)
    return [(seq, label) for seq in df.iloc[:, 1].dropna().tolist()]  # Remove NaN values

# Load datasets
therapeutic_csv = r"C:\Users\SURYA HA\OneDrive\Documents\Prediction of Therapeutic Peptide using Deep Learning and DAA\data\Therapeutic vs Non-Therapeutic Classification data\Therapeutic data\Therapeutic Data.csv"
non_therapeutic_csv = r"C:\Users\SURYA HA\OneDrive\Documents\Prediction of Therapeutic Peptide using Deep Learning and DAA\data\Therapeutic vs Non-Therapeutic Classification data\Non-Therapeutic data\Non-Therapeutic Data.csv"

therapeutic_sequences = load_sequences_from_csv(therapeutic_csv, label=0)
non_therapeutic_sequences = load_sequences_from_csv(non_therapeutic_csv, label=1)

# Combine datasets
data = therapeutic_sequences + non_therapeutic_sequences
df = pd.DataFrame(data, columns=["sequence", "label"])

# Extract features using ProtBERT with Progress Bar
X, times = [], []
print("\nExtracting features with ProtBERT (Using GPU)...")
start_time = time.time()

for seq in tqdm(df["sequence"], desc="Processing Sequences", unit="seq"):
    features, time_taken = extract_protbert_features(seq)
    X.append(features)
    times.append(time_taken)

end_time = time.time()
total_feature_time = end_time - start_time
print(f"\nTotal feature extraction time: {total_feature_time:.2f} seconds")
print(f"Average feature extraction time per sequence: {np.mean(times):.2f} seconds")

X = np.array(X)
y = df["label"].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)
joblib.dump(scaler, "scaler.pkl")  # Save scaler for later use

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Reshape data for CNN-LSTM model
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

# Build CNN-LSTM Model
model = Sequential([
    tf.keras.layers.Input(shape=(1024, 1)),  # Using ProtBERT embeddings (1024)
    
    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    Bidirectional(LSTM(64)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),
    
    Dense(2, activation='softmax')  # Two-class classification
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

# Train the model with Progress Bar
print("\nTraining model...")
start_time = time.time()

history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping, lr_scheduler], verbose=1)

end_time = time.time()
total_training_time = end_time - start_time
print(f"\nTotal training time: {total_training_time:.2f} seconds")

# Save trained model
model.save("model 1.h5")

# Function to classify a new protein sequence with time tracking
def classify_sequence(sequence):
    start_time = time.time()
    
    features, _ = extract_protbert_features(sequence)
    features = features.reshape(1, -1)
    
    scaler = joblib.load("scaler.pkl")  # Load saved scaler
    features = scaler.transform(features)
    features = np.expand_dims(features, axis=-1)
    
    prediction = model.predict(features)[0]
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nClassification time: {elapsed_time:.2f} seconds")

    class0, class1 = prediction[0], prediction[1]
    predicted_class = "Non-Therapeutic" if class0 > class1 else "Therapeutic"
    
    return {"Class0": class0, "Class1": class1, "Predicted": predicted_class}

Using device: cuda

Extracting features with ProtBERT (Using GPU)...


Processing Sequences:   0%|          | 0/57304 [00:00<?, ?seq/s]c:\Users\SURYA HA\anaconda3\envs\tf-gpu\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Processing Sequences: 100%|██████████| 57304/57304 [53:26<00:00, 17.87seq/s]  



Total feature extraction time: 3206.26 seconds
Average feature extraction time per sequence: 0.02 seconds

Training model...
Epoch 1/100
1433/1433 [==============================] - 131s 85ms/step - loss: 0.4682 - accuracy: 0.7678 - val_loss: 0.3680 - val_accuracy: 0.8142 - lr: 0.0010
Epoch 2/100
1433/1433 [==============================] - 116s 81ms/step - loss: 0.3767 - accuracy: 0.8185 - val_loss: 0.3362 - val_accuracy: 0.8406 - lr: 0.0010
Epoch 3/100
1433/1433 [==============================] - 89s 62ms/step - loss: 0.3404 - accuracy: 0.8391 - val_loss: 0.3159 - val_accuracy: 0.8549 - lr: 0.0010
Epoch 4/100
1433/1433 [==============================] - 96s 67ms/step - loss: 0.3204 - accuracy: 0.8486 - val_loss: 0.3075 - val_accuracy: 0.8573 - lr: 0.0010
Epoch 5/100
1433/1433 [==============================] - 81s 57ms/step - loss: 0.3075 - accuracy: 0.8561 - val_loss: 0.2960 - val_accuracy: 0.8653 - lr: 0.0010
Epoch 6/100
1433/1433 [==============================] - 82s 57ms/step -

In [2]:
# Example Test
test_sequence = "MNTYFISFITIIIFANGINGTSVDTSNKLLLQKANDFNLSQNLSSSRTRRTIANSFRIVGIRLEDETVETKNGIPTVLVDKEQQFRVFGSGLEENTAITFTNEKNDYGGPCLKPATDLFTPIEVSSNGFSALYSVKFPSFINEFFICAKTAEKTTNHSKAATTTPLEHQGNSDFLKIKTFEPLIPVWLAIIIIVTCLGFSALFSGLNLGLMSMDRTELKILRNTGTEKEKKYASKIAPVRDQGNYLLCSILLGNVLVNSTFTILLDGLTSGLFAVIFSTLAIVLFGEITPQAVCSRHGLAIGAKTILVTKTVMAITAPLSYPVSRILDKLLGEEIGNVYNRERLKELVRVTNDVNDLDKNEVNIISGALELRKKTVADVMTHINDAFMLSLDALLDFETVSEIMNSGYSRIPVYDGDRKNIVTLLYIKDLAFVDTDDNTPLKTLCEFYQNPVHFVFEDYTLDIMFNQFKEGTIGHIAFVHRVNNEGDGDPFYETVGLVTLEDVIEELIQAEIVDETDVFVDNRTKTRRNRYKKADFSAFAERREVQTVRISPQLTLATFQYLSTAVDAFKKDVISELILRRLLNQDVFHNIKTKGKSKDDPSLYIFTQGKAVDFFVLILEGRVEVTIGKEALMFESGPFTYFGTQALVPNVVIDSPTQMGSLQSLNMDSKIRQSFVPDYSVRAISDVIYITIKRVLYLTAKKATLLEKSRKSGTFSSETFDDEVERLLHSITENEKPSCFAQNQSTRRLSNRSINSSPTNMNRSPDFVYNSVDEAIQDDTKLKNIKHADNVTTSISLVAAELEDLHSGEQDTTAASMPLLPKLDDKFESKQSKP"   
result = classify_sequence(test_sequence)
# Therapeutic
print("Test Sequence Classification Result:")
print(result)

1/1 [==============================] - 1s 962ms/step

Classification time: 1.13 seconds
Test Sequence Classification Result:
{'Class0': 0.90984786, 'Class1': 0.09015216, 'Predicted': 'Non-Therapeutic'}


In [3]:
test_sequence = "MALVTKLLVASRNRKKLAELRRVLDGAGLSGLTLLSLGDVSPLPETPETGVTFEDNALAKARDAFSATGLASVADDSGLEVAALGGMPGVLSARWSGRYGDDAANTALLLAQLCDVPDERRGAAFVSACALVSGSGEVVVRGEWPGTIAREPRGDGGFGYDPVFVPYGDDRTAAQLSPAEKDAVSHRGRALALLLPALRSLATG"   
result = classify_sequence(test_sequence)
# Non-Therapeutic
print("Test Sequence Classification Result:")
print(result)

1/1 [==============================] - 0s 37ms/step

Classification time: 0.09 seconds
Test Sequence Classification Result:
{'Class0': 0.9996556, 'Class1': 0.0003443604, 'Predicted': 'Non-Therapeutic'}
